<a href="https://colab.research.google.com/github/fabrice237-prog/generative-ai-for-beginners/blob/main/Bienvenue_dans_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from typing import List, Tuple, Optional
from datetime import datetime

In [5]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

class Config:
    """Configuration du Modèle Transformer"""
    def __init__(self, **kwargs):
        self.vocab_size = 10000
        self.d_model = 256
        self.num_heads = 8
        self.num_layers = 6
        self.d_ff = 1024
        self.max_seq_len = 512
        self.dropout_rate = 0.1
        self.learning_rate = 1e-4
        self.batch_size = 32
        self.epochs = 50
        self.warmup_steps = 4000

        for key, value in kwargs.items():
            setattr(self, key, value)

# ==============================================================================
# TOKENIZER
# ==============================================================================

class SimpleTokenizer:
    """Tokenizer simple pour les conversations"""
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
        self.special_tokens = {
            '<PAD>': 0,
            '<UNK>': 1,
            '<EOS>': 2,
            '<BOS>': 3,
        }
        self.word_to_id = {k: v for k, v in self.special_tokens.items()}
        self.id_to_word = {v: k for k, v in self.special_tokens.items()}
        self.next_id = len(self.special_tokens)

    def train(self, texts: List[str]):
        """Construit le vocabulaire à partir des textes"""
        word_freq = {}
        for text in texts:
            words = text.lower().split()
            for word in words:
                word_freq[word] = word_freq.get(word, 0) + 1

        vocab_limit = self.vocab_size - len(self.special_tokens)
        sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:vocab_limit]

        for word, _ in sorted_words:
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1

        print(f"Vocabulaire entraîné : {len(self.word_to_id)} mots")

    def encode(self, text: str) -> List[int]:
        """Convertit un texte en IDs"""
        return [self.word_to_id.get(word, self.word_to_id['<UNK>']) for word in text.lower().split()]

    def decode(self, ids: List[int]) -> str:
        """Convertit des IDs en texte"""
        words = []
        for idx in ids:
            word = self.id_to_word.get(idx, '<UNK>')
            if word not in ['<PAD>', '<SEP>', '<EOS>', '<BOS>']:
                words.append(word)
        return " ".join(words)

    def save(self, path: str):
        """Sauvegarde le tokenizer"""
        data = {
            'word_to_id': self.word_to_id,
            'id_to_word': {str(k): v for k, v in self.id_to_word.items()},
            'vocab_size': self.vocab_size
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def load(self, path: str):
        """Charge un tokenizer sauvegardé"""
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(k): v for k, v in data['id_to_word'].items()}
        self.vocab_size = data['vocab_size']

# ==============================================================================
# MODÈLE TRANSFORMER
# ==============================================================================

class PositionalEncoding(tf.keras.layers.Layer):
    """Encodage positionnel sinusoïdal"""
    def __init__(self, max_seq_len: int, d_model: int, **kwargs):
        super().__init__(**kwargs)
        self.max_seq_len = max_seq_len
        self.d_model = d_model

        position = np.arange(max_seq_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

        pe = np.zeros((max_seq_len, d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)

        self.pe = tf.cast(pe[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

def create_causal_mask(size):
    """Crée un masque causal pour l'auto-attention"""
    return 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)

class MultiHeadAttention(tf.keras.layers.Layer):
    """Multi-Head Attention avec masque causal"""
    def __init__(self, d_model: int, num_heads: int, dropout_rate: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0

        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)

        output = tf.matmul(attention_weights, v)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        original_output_shape = tf.concat([tf.shape(output)[:2], [self.d_model]], axis=0)
        concat_attention = tf.reshape(output, original_output_shape)

        return self.dense(concat_attention)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = MultiHeadAttention(d_model, num_heads, dropout_rate)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, mask=None, training=False):
        attn_output = self.mha(x, x, x, mask, training=training)
        out1 = self.layernorm1(x + self.dropout1(attn_output, training=training))
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + self.dropout2(ffn_output, training=training))

class ConversationalTransformer(tf.keras.Model):
    """Modèle Transformer Conversationnel"""
    def __init__(self, config: Config, **kwargs):
        super().__init__(**kwargs)
        self.config = config
        self.embedding = tf.keras.layers.Embedding(config.vocab_size, config.d_model)
        self.pos_encoding = PositionalEncoding(config.max_seq_len, config.d_model)

        self.dec_blocks = [
            TransformerBlock(config.d_model, config.num_heads, config.d_ff, config.dropout_rate)
            for _ in range(config.num_layers)
        ]
        self.layernorm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.final_proj = tf.keras.layers.Dense(config.vocab_size)
        self.dropout = tf.keras.layers.Dropout(config.dropout_rate)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.config.d_model, tf.float32))
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)

        mask = create_causal_mask(seq_len)
        for block in self.dec_blocks:
            x = block(x, mask, training=training)

        x = self.layernorm(x)
        return self.final_proj(x)

    def generate(self, input_ids, max_new_tokens=50, temperature=0.7, top_k=50):
        """Génère une réponse auto-régressive"""
        generated = list(input_ids[0] if isinstance(input_ids, np.ndarray) else input_ids)

        for _ in range(max_new_tokens):
            context = np.array([generated[-self.config.max_seq_len:]])
            logits = self(context, training=False)
            next_token_logits = logits[0, -1, :] / temperature

            if top_k > 0:
                values, indices = tf.math.top_k(next_token_logits, k=top_k)
                indices = indices.numpy()
                min_val = tf.reduce_min(values)
                next_token_logits = tf.where(
                    next_token_logits < min_val,
                    tf.ones_like(next_token_logits) * -float('inf'),
                    next_token_logits
                )

            probs = tf.nn.softmax(next_token_logits, axis=-1)
            next_token = tf.random.categorical(tf.expand_dims(probs, 0), num_samples=1)[0, 0].numpy()

            if next_token == 2:  # <EOS>
                break
            generated.append(next_token)

        return generated

# ==============================================================================
# DATASET
# ==============================================================================

class ConversationDataset:
    """Dataset pour l'entraînement"""
    def __init__(self, conversations: List[Tuple[str, str]], tokenizer: SimpleTokenizer, max_seq_len: int):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.data = []

        for context, response in conversations:
            context_ids = tokenizer.encode(context)
            response_ids = tokenizer.encode(response)
            input_ids = context_ids + [tokenizer.special_tokens['<BOS>']] + response_ids + [tokenizer.special_tokens['<EOS>']]

            target_ids = input_ids[1:] + [tokenizer.special_tokens['<PAD>']]

            if len(input_ids) > max_seq_len:
                input_ids = input_ids[:max_seq_len]
                target_ids = target_ids[:max_seq_len]
            else:
                padding_len = max_seq_len - len(input_ids)
                input_ids += [tokenizer.special_tokens['<PAD>']] * padding_len
                target_ids += [tokenizer.special_tokens['<PAD>']] * padding_len

            self.data.append((np.array(input_ids, dtype=np.int32), np.array(target_ids, dtype=np.int32)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def create_training_data() -> List[Tuple[str, str]]:
    """Génère des données d'exemple pour l'entraînement"""
    # conversations = [
    #     # Salutations
    #     ("Bonjour !", "Bonjour ! Comment puis-je vous aider ?"),
    #     ("Salut", "Salut ! Comment allez-vous ?"),
    #     ("Bonsoir", "Bonsoir ! Quelle belle soirée !"),
    #     # Présentations
    #     ("Comment tu t'appelles ?", "Je suis un assistant IA conversationnel."),
    #     ("Qui es-tu ?", "Je suis un chatbot basé sur l'architecture Transformer."),
    #     # Questions simples
    #     ("Quel âge as-tu ?", "Je suis un programme informatique, je n'ai pas d'âge."),
    #     ("Où habites-tu ?", "Dans le cloud, disponible 24h/24 !"),
    #     # Météo
    #     ("Quel temps fait-il ?", "Je ne peux pas vérifier la météo en temps réel."),
    #     # Aide
    #     ("Peux-tu m'aider ?", "Bien sûr ! De quoi avez-vous besoin ?"),
    #     # Remerciements
    #     ("Merci", "Je vous en prie !"),
    #     ("Merci beaucoup", "Avec plaisir !"),
    #     # Au revoir
    #     ("Au revoir", "Au revoir ! Passez une excellente journée !"),
    #     ("Bye", "Au revoir !"),
    # ]
    conversations = []
    with open('data.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    for dico in data:
        conversations.append((" ".join(dico['user_input']), dico['bot_response']))

    # # Augmentation des données basique
    # augmented = []
    # for ctx, resp in conversations:
    #     augmented.append((ctx, resp))
    #     if "bonjour" in ctx.lower():
    #         augmented.append((ctx.replace("Bonjour", "Salut"), resp))

    return conversations

# ==============================================================================
# ENTRAÎNEUR (TRAINER)
# ==============================================================================

class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Schedule d'apprentissage avec warmup"""
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

class ChatbotTrainer:
    def __init__(self, config: Config):
        self.config = config
        self.device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
        print(f"🌍 Utilisation du périphérique : {self.device}")

        with tf.device(self.device):
            self.model = ConversationalTransformer(config)

        learning_rate = CustomSchedule(config.d_model, config.warmup_steps)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
        self.train_loss = tf.keras.metrics.Mean(name='train_loss')

    def loss_function(self, real, pred):
        """Fonction de perte avec masque"""
        loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
        loss_ = loss_object(real, pred)
        mask = tf.math.logical_not(tf.math.equal(real, 0))
        mask = tf.cast(mask, dtype=loss_.dtype)
        loss_ *= mask
        return tf.reduce_sum(loss_) / (tf.reduce_sum(mask) + 1e-8)

    @tf.function
    def train_step(self, inputs, targets):
        """Une étape d'entraînement"""
        with tf.GradientTape() as tape:
            predictions = self.model(inputs, training=True)
            loss = self.loss_function(targets, predictions)

        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))
        self.train_loss.update_state(loss)
        return loss

    def train(self, dataset: ConversationDataset, epochs=None, save_every=5):
        if epochs is None:
            epochs = self.config.epochs

        print(f"🔥 Entraînement commencé pour {epochs} époques...")

        # Préparation du générateur pour tf.data
        def gen():
            for i in range(len(dataset)):
                yield dataset[i]

        tf_dataset = tf.data.Dataset.from_generator(
            gen,
            output_signature=(
                tf.TensorSpec(shape=(self.config.max_seq_len,), dtype=tf.int32),
                tf.TensorSpec(shape=(self.config.max_seq_len,), dtype=tf.int32)
            )
        ).shuffle(1000).batch(self.config.batch_size).prefetch(tf.data.AUTOTUNE)

        for epoch in range(epochs):
            self.train_loss.reset_state()

            for batch, (inputs, targets) in enumerate(tf_dataset):
                with tf.device(self.device):
                    batch_loss = self.train_step(inputs, targets)

                if batch % 10 == 0:
                    # print(f"Époque {epoch+1} | Batch {batch} | Perte: {batch_loss.numpy():.4f} | LR: {self.optimizer.learning_rate(self.optimizer.iterations).numpy():.6f}")
                    print(f"Époque {epoch+1} | Batch {batch} | Perte: {batch_loss.numpy():.4f} | LR: {float(self.optimizer.learning_rate)}")

            print(f"✨ Époque {epoch+1} terminée | Perte moyenne : {self.train_loss.result().numpy():.4f}")

            if (epoch + 1) % save_every == 0 or (epoch + 1) == epochs:
                self.save_checkpoint(epoch + 1, self.train_loss.result().numpy())

    def save_checkpoint(self, epoch, loss):
        os.makedirs("models", exist_ok=True)
        path = f"models/chatbot_transformer_epoch_{epoch}.weights.h5"
        self.model.save_weights(path)
        print(f"💾 Modèle sauvegardé : {path}")

    def load_checkpoint(self, path):
        # Initialisation des poids du modèle avec un batch fictif avant chargement
        dummy_inputs = tf.zeros((1, self.config.max_seq_len), dtype=tf.int32)
        self.model(dummy_inputs, training=False)
        self.model.load_weights(path)
        print(f"✅ Modèle chargé depuis {path}")

# ==============================================================================
# INTERFACE INTERACTIVE (CHAT)
# ==============================================================================

class ChatbotInterface:
    def __init__(self, tokenizer_path: str, model_path: Optional[str] = None, config: Optional[Config] = None):
        self.config = config if config else Config()
        self.tokenizer = SimpleTokenizer()
        self.tokenizer.load(tokenizer_path)

        self.device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
        with tf.device(self.device):
            self.model = ConversationalTransformer(self.config)

        if model_path and os.path.exists(model_path):
            # Initialisation nécessaire avant chargement des poids
            dummy_inputs = tf.zeros((1, self.config.max_seq_len), dtype=tf.int32)
            self.model(dummy_inputs, training=False)
            self.model.load_weights(model_path)
            print("✅ Modèle chargé avec succès.")
        else:
            print("⚠️ Modèle non entraîné détecté (réponses aléatoires).")

        self.conversation_history = []

    def generate_response(self, user_input: str, temperature: float = 0.7, max_len: int = 60) -> str:
        self.conversation_history.append(f"Utilisateur: {user_input}")

        if len(self.conversation_history) > 10:
            self.conversation_history = self.conversation_history[-10:]

        context_text = " ".join(self.conversation_history)
        input_ids = self.tokenizer.encode(context_text)

        if len(input_ids) > self.config.max_seq_len - 2:
            input_ids = input_ids[-(self.config.max_seq_len - 2):]

        # Ajout du token de début de génération
        input_ids = [self.tokenizer.special_tokens['<BOS>']] + input_ids
        input_tensor = np.array([input_ids], dtype=np.int32)

        with tf.device(self.device):
            generated_ids = self.model.generate(
                input_tensor,
                max_new_tokens=max_len,
                temperature=temperature
            )

        response = self.tokenizer.decode(generated_ids[len(input_ids):])
        if not response.strip():
            response = "Je ne suis pas sûr de comprendre. Pourriez-vous reformuler ?"

        self.conversation_history.append(f"Assistant: {response}")
        return response

    def chat(self):
        print("\n" + "="*60)
        print("💬 CHATBOT TRANSFORMER ACTIF - PRÊT À DISCUTER")
        print("Commandes : 'quit' pour quitter, 'reset' pour effacer la mémoire.")
        print("="*60)

        temperature = 0.7
        while True:
            try:
                user_input = input("\n👤 Vous : ").strip()
                if not user_input:
                    continue

                if user_input.lower() in ['quit', 'exit']:
                    print("👋 Au revoir !")
                    break

                if user_input.lower() == 'reset':
                    self.conversation_history = []
                    print("🔄 Mémoire réinitialisée !")
                    continue

                if user_input.lower().startswith('temp '):
                    try:
                        temperature = float(user_input.split()[1])
                        print(f"🌡️ Température réglée sur {temperature}")
                        continue
                    except:
                        print("Erreur de format de température.")
                        continue

                response = self.generate_response(user_input, temperature=temperature)
                print(f"🤖 IA : {response}")

            except KeyboardInterrupt:
                print("\n👋 Au revoir !")
                break
            except Exception as e:
                print(f"❌ Une erreur est survenue : {e}")

# ==============================================================================
# PIPELINE PRINCIPALE (MAIN)
# ==============================================================================

def train_model():
    config = Config(batch_size=4, epochs=5)  # Paramètres réduits pour test rapide
    tokenizer = SimpleTokenizer(vocab_size=config.vocab_size)

    print("📋 Préparation des données...")
    conversations = create_training_data()
    all_texts = [ctx + " " + resp for ctx, resp in conversations]

    tokenizer.train(all_texts)
    tokenizer.save("chatbot_save/tokenizer.json")

    dataset = ConversationDataset(conversations, tokenizer, config.max_seq_len)
    trainer = ChatbotTrainer(config)
    trainer.train(dataset)

if __name__ == "__main__":
    # Décommente cette ligne si tu veux lancer un entraînement complet :
    train_model()

    # Mode Chat (nécessite l'entraînement préalable et la sauvegarde des fichiers)
    if os.path.exists("chatbot_save/tokenizer.json"):
        # Trouve le dernier checkpoint si disponible
        checkpoint_dir = "chatbot_save"
        h5_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.h5')]
        latest_model = os.path.join(checkpoint_dir, sorted(h5_files)[-1]) if h5_files else None

        bot = ChatbotInterface(tokenizer_path="chatbot_save/tokenizer.json", model_path=latest_model)
        bot.chat()
    else:
        print("🚀 Aucun tokenizer trouvé. Lancement d'un entraînement d'exemple complet...")
        train_model()
        print("\n✅ Entraînement initial terminé ! Relance le script pour chatter.")


📋 Préparation des données...
Vocabulaire entraîné : 5591 mots
🌍 Utilisation du périphérique : /CPU:0
🔥 Entraînement commencé pour 5 époques...
Époque 1 | Batch 0 | Perte: 9.2144 | LR: 2.470529523179721e-07
Époque 1 | Batch 10 | Perte: 9.2129 | LR: 2.7175824470759835e-06
Époque 1 | Batch 20 | Perte: 9.1587 | LR: 5.188112027099123e-06
Époque 1 | Batch 30 | Perte: 9.0731 | LR: 7.658641152374912e-06
Époque 1 | Batch 40 | Perte: 9.0410 | LR: 1.0129170732398052e-05
Époque 1 | Batch 50 | Perte: 8.9984 | LR: 1.2599700312421191e-05
Époque 1 | Batch 60 | Perte: 9.0193 | LR: 1.5070229892444331e-05
Époque 1 | Batch 70 | Perte: 9.0119 | LR: 1.7540760381962173e-05
Époque 1 | Batch 80 | Perte: 8.9178 | LR: 2.0011289961985312e-05
✨ Époque 1 terminée | Perte moyenne : 9.0679
Époque 2 | Batch 0 | Perte: 8.8559 | LR: 2.0505394786596298e-05
Époque 2 | Batch 10 | Perte: 8.8244 | LR: 2.2975924366619438e-05
Époque 2 | Batch 20 | Perte: 8.6999 | LR: 2.5446453946642578e-05
Époque 2 | Batch 30 | Perte: 8.6539 |